In [0]:
import requests
import pandas as pd
from datetime import datetime
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

In [ ]:
env = dbutils.widgets.get("env")

In [0]:
import requests
import pandas as pd
import time
from datetime import datetime

API_KEY = dbutils.secrets.get( scope="aqi-scope", key="aqi-api-key" )

BASE_URL = "https://api.data.gov.in/resource/3b01bcb8-0b14-4abf-b6f2-c1bfd384ba69"

limit = 100
offset = 0

all_records = []

headers = {
    "User-Agent": "Mozilla/5.0"
}

while True:

    params = {
        "api-key": API_KEY,
        "format": "json",
        "limit": limit,
        "offset": offset
    }

    retry = 3
    data_finished = False
    for attempt in range(retry):

        try:

            response = requests.get(
                BASE_URL,
                params=params,
                headers=headers,
                timeout=30
            )

            print(f"Status Code: {response.status_code}")
            if response.status_code == 200:
                data = response.json()
                records = data.get("records", [])

                # STOP CONDITION
                if not records:
                    print("No more records found")
                    data_finished = True
                    break

                all_records.extend(records)
                print(f"Fetched {len(records)} | Total: {len(all_records)}")
                offset += limit
                time.sleep(1)
                break

            else:
                print(f"Retry {attempt+1} -> Status: {response.status_code}")
                time.sleep(5)

        except Exception as e:
            print(f"Error: {e}")
            time.sleep(5)

    # OUTER LOOP BREAK
    if data_finished:
        break

print(f"\nTotal Records Collected: {len(all_records)}")

In [0]:
df = spark.createDataFrame(all_records)
df.write.mode("append").saveAsTable(f"{env}_catalog.aqi_strm_{env}.aqi")